# Etapa 3 — Estratégia de Validação e Modelos de Referência (Baselines)

Neste notebook, implementamos:
1. **Divisão Treino/Teste por Grupos (`GroupShuffleSplit`)**: garantindo que nenhum estudante apareça simultaneamente no conjunto de treinamento e teste.
2. **Definição dos Baselines Obrigatórios**:
   - **Baseline 1 (Ganho Zero)**: previsão $\hat{y} = 0$.
   - **Baseline 2 (Média do Treino)**: previsão $\hat{y} = \bar{y}_{treino}$.
3. **Avaliação dos Baselines** nos conjuntos de teste independentes.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from wordgen.modelagem.preparacao import carregar_e_preparar_dados
from wordgen.modelagem.validacao import dividir_treino_teste_agrupado
from wordgen.modelagem.baselines import BaselineGanhoZero, BaselineMediaTreino
from wordgen.modelagem.avaliacao import avaliar_modelo, consolidar_tabela_metricas

DATA_DIR = Path("data") if Path("data").exists() else Path("../../data")
datasets = carregar_e_preparar_dados(DATA_DIR)
cenarios = [
    ("Vocabulário — Exp A", "vocabulario_exp_a"),
    ("Vocabulário — Exp B", "vocabulario_exp_b"),
    ("TDE II — Exp A", "tde_exp_a"),
    ("TDE II — Exp B", "tde_exp_b"),
]

resultados_baselines = []

for rotulo, chave in cenarios:
    df = datasets[chave]
    X_train, X_test, y_train, y_test, g_train, g_test = dividir_treino_teste_agrupado(
        df, target_col="ganho", grupo_col="aluno_id", test_size=0.20, random_state=42
    )

    # 1. Baseline Zero
    b0 = BaselineGanhoZero().fit(X_train, y_train)
    diag0 = avaliar_modelo(b0, X_test, y_test, "Baseline 1 (Ganho Zero)", rotulo)
    resultados_baselines.append(diag0)

    # 2. Baseline Média Treino
    bm = BaselineMediaTreino().fit(X_train, y_train)
    diagm = avaliar_modelo(bm, X_test, y_test, "Baseline 2 (Média Treino)", rotulo)
    resultados_baselines.append(diagm)

df_base_res = consolidar_tabela_metricas(resultados_baselines)
display(df_base_res)

## 3.1. Interpretação dos Baselines
- Como esperado, o $R^2$ de ambos os baselines de referência é próximo de zero ou ligeiramente negativo, pois um preditor constante não captura a variância intra-aluno.
- O MAE do Baseline da Média serve como o patamar de referência mínimo que os modelos supervisionados (Random Forest e MLP) devem superar.